# FAISS- VectorDB

Facebook AI Similarity Search (FAISS) is a library for efficient similarity seatch and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and paramater tuning

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS


from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader(file_path="speech.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size = 200, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

Created a chunk of size 403, which is longer than the specified 200
Created a chunk of size 615, which is longer than the specified 200
Created a chunk of size 589, which is longer than the specified 200


In [2]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a language's lexicon. There are many different intentional speech acts, such as informing, declaring, asking, persuading, directing; acts may vary in various aspects like enunciation, intonation, loudness, and tempo to convey meaning."),
 Document(metadata={'source': 'speech.txt'}, page_content='Individuals may also unintentionally communicate aspects of their social position through speech, such as sex, age, place of origin, physiological and mental condition, education, and experiences.'),
 Document(metadata={'source': 'speech.txt'}, page_content="While normally used to facilitate communication with others, people may also use speech without the intent to communicate. Speech may nevertheless express emotions or desires; people talk to themselves sometimes in ac

In [4]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")
db = FAISS.from_documents(docs, embeddings)
db


In [ ]:
### Querying 

query = "otolaryngology"

docs = db.similarity_search(query)
docs[0].page_content

'Speech compares with written language,[1] which may differ in its vocabulary, syntax, and phonetics from the spoken language, a situation called diglossia.'

# As a Retriever

We can also convert the vectorstore into a Retriever Class. This allows us to easily use it in other LangChain methods, which largely work with retrievers.

In [11]:
retriever = db.as_retriever()
retriever.invoke(query)

[Document(id='99b430b9-9482-4a7b-b914-905486335934', metadata={'source': 'speech.txt'}, page_content="Researchers study many different aspects of speech: speech production and speech perception of the sounds used in a language, speech repetition, speech errors, the ability to map heard spoken words onto the vocalizations needed to recreate them, which plays a key role in children's enlargement of their vocabulary, and what different areas of the human brain, such as Broca's area and Wernicke's area, underlie speech. Speech is the subject of study for linguistics, cognitive science, communication studies, psychology, computer science, speech pathology, otolaryngology, and acoustics."),
 Document(id='e3fee98d-8d42-4cc3-a9f3-94945c0303e8', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a language's lexicon. There are many

In [12]:
docs[0].page_content

"Researchers study many different aspects of speech: speech production and speech perception of the sounds used in a language, speech repetition, speech errors, the ability to map heard spoken words onto the vocalizations needed to recreate them, which plays a key role in children's enlargement of their vocabulary, and what different areas of the human brain, such as Broca's area and Wernicke's area, underlie speech. Speech is the subject of study for linguistics, cognitive science, communication studies, psychology, computer science, speech pathology, otolaryngology, and acoustics."

# Similarity Search with score

There are some specific FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query given to them. The returned distance score is L2 distance (Manhattan Distance). Therefore, a lower score is better

In [13]:
docs_and_sscore = db.similarity_search_with_score(query)

docs_and_sscore

[(Document(id='99b430b9-9482-4a7b-b914-905486335934', metadata={'source': 'speech.txt'}, page_content="Researchers study many different aspects of speech: speech production and speech perception of the sounds used in a language, speech repetition, speech errors, the ability to map heard spoken words onto the vocalizations needed to recreate them, which plays a key role in children's enlargement of their vocabulary, and what different areas of the human brain, such as Broca's area and Wernicke's area, underlie speech. Speech is the subject of study for linguistics, cognitive science, communication studies, psychology, computer science, speech pathology, otolaryngology, and acoustics."),
  np.float32(0.7388837)),
 (Document(id='e3fee98d-8d42-4cc3-a9f3-94945c0303e8', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a langua

In [14]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.03156096488237381,
 -0.00799549836665392,
 -0.042591750621795654,
 0.010466402396559715,
 -0.022258922457695007,
 -0.038654875010252,
 -0.020039094612002373,
 0.018486706539988518,
 0.017388202250003815,
 0.022858193144202232,
 0.02439752221107483,
 -0.02879156358540058,
 0.009637273848056793,
 -0.006977500393986702,
 -0.029741646721959114,
 0.012896985746920109,
 -0.0013432676205411553,
 -0.0021476084366440773,
 0.003683018032461405,
 -0.02593396231532097,
 -0.008157090283930302,
 0.027343446388840675,
 -0.06498868763446808,
 0.05248018354177475,
 -0.047554418444633484,
 -0.004639827646315098,
 -0.032133933156728745,
 0.006802856922149658,
 0.04082697629928589,
 0.06017023324966431,
 -0.006624414585530758,
 0.005279898643493652,
 -0.006519422400742769,
 -0.045523934066295624,
 -0.0013466879026964307,
 -0.06044130027294159,
 0.05436696484684944,
 -0.056245673447847366,
 0.022669315338134766,
 -0.028664007782936096,
 0.016778409481048584,
 -0.010777565650641918,
 0.018407583236694336

In [16]:
search_result = db.similarity_search_by_vector(embedding_vector)

search_result

[Document(id='99b430b9-9482-4a7b-b914-905486335934', metadata={'source': 'speech.txt'}, page_content="Researchers study many different aspects of speech: speech production and speech perception of the sounds used in a language, speech repetition, speech errors, the ability to map heard spoken words onto the vocalizations needed to recreate them, which plays a key role in children's enlargement of their vocabulary, and what different areas of the human brain, such as Broca's area and Wernicke's area, underlie speech. Speech is the subject of study for linguistics, cognitive science, communication studies, psychology, computer science, speech pathology, otolaryngology, and acoustics."),
 Document(id='e3fee98d-8d42-4cc3-a9f3-94945c0303e8', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a language's lexicon. There are many

In [18]:
# Saving and Loading

db.save_local("faiss_local")



In [27]:
new_db = FAISS.load_local("faiss_local", embeddings, allow_dangerous_deserialization=True)

docs = new_db.similarity_search(query)

In [28]:
docs

[Document(id='99b430b9-9482-4a7b-b914-905486335934', metadata={'source': 'speech.txt'}, page_content="Researchers study many different aspects of speech: speech production and speech perception of the sounds used in a language, speech repetition, speech errors, the ability to map heard spoken words onto the vocalizations needed to recreate them, which plays a key role in children's enlargement of their vocabulary, and what different areas of the human brain, such as Broca's area and Wernicke's area, underlie speech. Speech is the subject of study for linguistics, cognitive science, communication studies, psychology, computer science, speech pathology, otolaryngology, and acoustics."),
 Document(id='e3fee98d-8d42-4cc3-a9f3-94945c0303e8', metadata={'source': 'speech.txt'}, page_content="Speech is the use of the human voice as a medium for language. Spoken language combines vowel and consonant sounds to form units of meaning like words, which belong to a language's lexicon. There are many